# Steering Factory — Colab runner
This notebook clones the GitHub repository, runs the reusable package on Colab disk, and publishes each finalized artifact to Google Drive.

In [ ]:
import os
import sys

REPO_URL = 'https://github.com/UtkarshC99/steering-factory'
REPO_DIR = '/content/steering-factory'

if os.path.isdir(f'{REPO_DIR}/.git'):
    print('Repo already cloned — pulling latest...')
    !git -C {REPO_DIR} pull --ff-only
else:
    print('Cloning repo...')
    !git clone {REPO_URL} {REPO_DIR}

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
%cd {REPO_DIR}

In [ ]:
%cd /content/steering-factory
!pip -q install -r requirements.txt

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Write actively to fast Colab disk; finalized runs are mirrored to Drive.
ARTIFACT_ROOT = '/content/steering_factory_artifacts'
PUBLISH_ROOT = '/content/drive/MyDrive/Projects/steering-factory'
MANIFEST = 'manifests/default_colab.yaml'

In [ ]:
!python -m steering_factory prepare-data $MANIFEST --set artifacts.root=$ARTIFACT_ROOT --set artifacts.publish_root=$PUBLISH_ROOT

## Two ways to run the steering/QLoRA arms

**Live (recommended in this notebook)** — cells below call `steering_factory.runner` directly, in-process, and pass
a callback that updates a plot in place every few generations/training steps. This needs the in-process call because
a `plotly.FigureWidget` can only be mutated live from the same kernel that displayed it — a subprocess's stdout
can't drive a widget.

**Headless / CLI (kept for scripted or non-notebook use)** — the `run_sf` helper below shells out to
`python -m steering_factory ...` exactly like the CLI docs describe, with no live plot, just streamed text output.
Use this if you're running unattended, on a machine without ipywidgets, or want the exact command you'd put in a
script. It is not removed or deprecated — swap a live cell for its headless equivalent any time by calling `run_sf`
with the same arguments.

In [ ]:
import json
import subprocess

def run_sf(*args):
    """Headless/CLI path: runs `python -m steering_factory <args>` as a
    subprocess, streams output live as text, and returns the parsed
    {run_id, artifact_dir, status} dict from the final JSON line. Raises on
    a non-zero exit or a non-completed run status. No live plot -- for that,
    use the in-process cells below instead."""
    cmd = ['python', '-m', 'steering_factory', *args]
    print('$', ' '.join(cmd))
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    lines = []
    for line in proc.stdout:
        print(line, end='')
        lines.append(line.rstrip('\n'))
    returncode = proc.wait()
    if returncode != 0:
        raise RuntimeError(f'steering_factory {args[0]} exited with code {returncode}')
    payload = None
    for line in reversed(lines):
        try:
            payload = json.loads(line)
            break
        except json.JSONDecodeError:
            continue
    if payload is None:
        raise RuntimeError(f'Could not find a JSON result line in steering_factory {args[0]} output.')
    if payload.get('status') not in (None, 'completed'):
        raise RuntimeError(f"steering_factory {args[0]} finished with status={payload.get('status')!r}: {payload}")
    return payload

## Steering arm (live)

Runs the full extract + evaluate grid (Pillar 1 safety/behavior recipes and Pillar 3 ablation grid) in one artifact,
in-process, with a live plot of perplexity-ratio and JS-divergence-vs-baseline that updates every `update_every`
generations. For faster iteration on extraction methods or layers alone, call `run_extract` then `run_evaluate`
instead of `run_steering` — same live-callback pattern applies to both.

Prefer the headless version? Replace this cell's body with:
```python
steering_result = run_sf('run', MANIFEST, '--set', f'artifacts.root={ARTIFACT_ROOT}', '--set', f'artifacts.publish_root={PUBLISH_ROOT}')
STEERING_RUN_DIR = steering_result['artifact_dir']
```

In [ ]:
from IPython.display import display

from steering_factory.manifest import load_manifest
from steering_factory.runner import run_steering
from steering_factory.callbacks import CompositeCallback
from steering_factory.live_plot import SteeringGridPlot, BiPOConvergencePlot

steering_manifest = load_manifest(MANIFEST, [f'artifacts.root={ARTIFACT_ROOT}', f'artifacts.publish_root={PUBLISH_ROOT}'])

steering_plot = SteeringGridPlot(update_every=5)
bipo_plot = BiPOConvergencePlot(update_every=1)
display(steering_plot.figure)
display(bipo_plot.figure)
live_callback = CompositeCallback([steering_plot, bipo_plot])

# The durable per-row record (results/generations.jsonl, vectors/index.jsonl,
# telemetry.json) is written by run_steering itself regardless of callback;
# the callback here only drives the two live plots above. For an incremental
# on-disk progress trace as well (useful if a run might get interrupted),
# attach a JsonlCallback(store, arm=...) in the split extract/evaluate
# pattern instead, which has access to the store before the grid runs.
store = run_steering(steering_manifest, command='notebook: run_steering (live)', callback=live_callback)
STEERING_RUN_DIR = str(store.path)
print('\nSteering run:', STEERING_RUN_DIR, '| status:', store.artifact.status)

## QLoRA arm (live)

Trains one matched QLoRA adapter per model/recipe steer split, then evaluates each adapter on the same held-out
validation/test examples the steering arm used — required for `compare` to produce a matched quality/cost report.
This is the slowest cell in the notebook; the loss/grad-norm plot updates every `logging_steps` (from the manifest's
`finetune` block), forwarded straight from `transformers.Trainer`'s own instrumentation.

Prefer the headless version? Replace this cell's body with:
```python
qlora_result = run_sf('finetune', MANIFEST, '--set', f'artifacts.root={ARTIFACT_ROOT}', '--set', f'artifacts.publish_root={PUBLISH_ROOT}')
QLORA_RUN_DIR = qlora_result['artifact_dir']
```

In [ ]:
from steering_factory.runner import run_qlora
from steering_factory.live_plot import QLoRALossPlot

qlora_manifest = load_manifest(MANIFEST, [f'artifacts.root={ARTIFACT_ROOT}', f'artifacts.publish_root={PUBLISH_ROOT}'])

qlora_plot = QLoRALossPlot(update_every=1)
display(qlora_plot.figure)

qlora_store = run_qlora(qlora_manifest, command='notebook: run_qlora (live)', callback=qlora_plot)
QLORA_RUN_DIR = str(qlora_store.path)
print('\nQLoRA run:', QLORA_RUN_DIR, '| status:', qlora_store.artifact.status)

## Compare

Joins the steering run and the QLoRA run on `(model_id, recipe_id)`: the best steering config is selected on
validation only, held-out test quality is reported for both arms, and cost (wall time, labeled-example count,
artifact size, ms/token) is normalized across both. Writes `report.json` and `report.md` under `--output-root`.

This step is quick post-processing (no model load), so it stays a CLI subprocess call regardless of which arm
cells you ran above — `STEERING_RUN_DIR`/`QLORA_RUN_DIR` are set the same way by either the live or headless cells.

In [ ]:
COMPARISON_OUTPUT = f'{ARTIFACT_ROOT}/comparisons'

!python -m steering_factory compare $STEERING_RUN_DIR $QLORA_RUN_DIR --output-root $COMPARISON_OUTPUT

In [ ]:
from IPython.display import Markdown, display

with open(f'{COMPARISON_OUTPUT}/report.md', encoding='utf-8') as f:
    display(Markdown(f.read()))